In [2]:
%%writefile /content/app.py
import joblib
import pandas as pd
import streamlit as st
from pathlib import Path

st.set_page_config(page_title="EMI Eligibility Predictor",
                   page_icon="💰", layout="wide")

# ------------------ load artifacts ------------------
@st.cache_resource
def load_artifacts(path="/content/emi_artifacts.joblib"):
    if not Path(path).exists():
        st.error(f"Artifacts file not found: {path}")
        st.stop()
    return joblib.load(path)

A = load_artifacts()
classifier      = A["classifier"]
regressor       = A["regressor"]
scaler          = A["scaler"]
label_encoder   = A["label_encoder"]
feature_columns = A["feature_columns"]
num_cols        = A["num_cols"]
cat_cols        = A["cat_cols"]
category_values = A["category_values"]

# ------------------ UI ------------------
st.title("💰 EMI Eligibility Predictor")
st.caption("Enter applicant details in the sidebar and click Predict.")

st.sidebar.header("Applicant Details")
with st.sidebar.form("applicant_form"):

    st.subheader("Personal")
    age            = st.number_input("Age", 18, 80, 42)
    gender         = st.selectbox("Gender",         category_values["gender"])
    marital_status = st.selectbox("Marital Status", category_values["marital_status"])
    education      = st.selectbox("Education",      category_values["education"])
    family_size    = st.number_input("Family size", 1, 15, 4)
    dependents     = st.number_input("Dependents",  0, 10, 1)

    st.subheader("Employment")
    employment_type     = st.selectbox("Employment type", category_values["employment_type"])
    company_type        = st.selectbox("Company type",    category_values["company_type"])
    years_of_employment = st.number_input("Years of employment", 0, 50, 10)

    st.subheader("Income & Savings")
    monthly_salary = st.number_input("Monthly salary",  0, value=70000, step=1000)
    bank_balance   = st.number_input("Bank balance",    0, value=300000, step=1000)
    emergency_fund = st.number_input("Emergency fund",  0, value=80000,  step=1000)
    credit_score   = st.number_input("Credit score",    300, 850, 650)

    st.subheader("Monthly Expenses")
    house_type             = st.selectbox("House type", category_values["house_type"])
    monthly_rent           = st.number_input("Monthly rent",          0, 50000, 10000, step=500)
    school_fees            = st.number_input("School fees",           0, 20000, 7000,  step=500)
    college_fees           = st.number_input("College fees",          0, 50000, 0,     step=500)
    travel_expenses        = st.number_input("Travel expenses",       500, 15000, 7000, step=500)
    groceries_utilities    = st.number_input("Groceries & utilities", 3000, 30000, 16000, step=500)
    other_monthly_expenses = st.number_input("Other expenses",        0, 20000, 10000, step=500)

    st.subheader("Existing Loans")
    existing_loans     = st.number_input("Existing loans", 0, 10, 0)
    current_emi_amount = st.number_input("Current EMI amount", 0, 80000, 0, step=500)

    st.subheader("Loan Request")
    emi_scenario     = st.selectbox("EMI scenario", category_values["emi_scenario"])
    requested_amount = st.number_input("Requested amount",   10000, 1500000, 250000, step=5000)
    requested_tenure = st.number_input("Requested tenure (months)", 3, 84, 24)

    submitted = st.form_submit_button("🔮 Predict")

# ------------------ preprocessing ------------------
def build_feature_row():
    row = {
        "age": age, "gender": gender, "marital_status": marital_status,
        "education": education, "employment_type": employment_type,
        "company_type": company_type, "house_type": house_type,
        "emi_scenario": emi_scenario,
        "monthly_salary": monthly_salary,
        "years_of_employment": years_of_employment,
        "monthly_rent": monthly_rent,
        "family_size": family_size, "dependents": dependents,
        "school_fees": school_fees, "college_fees": college_fees,
        "travel_expenses": travel_expenses,
        "groceries_utilities": groceries_utilities,
        "other_monthly_expenses": other_monthly_expenses,
        "existing_loans": existing_loans,
        "current_emi_amount": current_emi_amount,
        "credit_score": credit_score,
        "bank_balance": bank_balance,
        "emergency_fund": emergency_fund,
        "requested_amount": requested_amount,
        "requested_tenure": requested_tenure,
    }
    X = pd.DataFrame([row])

    # engineered features (same as Step 4)
    X["total_expenses"] = (X["monthly_rent"] + X["school_fees"] + X["college_fees"]
                           + X["travel_expenses"] + X["groceries_utilities"]
                           + X["other_monthly_expenses"] + X["current_emi_amount"])
    X["disposable_income"]       = X["monthly_salary"] - X["total_expenses"]
    X["debt_to_income"]          = X["current_emi_amount"] / X["monthly_salary"]
    X["expense_to_income"]       = X["total_expenses"] / X["monthly_salary"]
    X["savings_ratio"]           = X["bank_balance"] / (X["monthly_salary"] * 12)
    X["requested_emi_estimate"]  = X["requested_amount"] / X["requested_tenure"]
    X["requested_emi_to_income"] = X["requested_emi_estimate"] / X["monthly_salary"]

    # one-hot encode, align columns with training, scale
    X = pd.get_dummies(X, columns=cat_cols, drop_first=True)
    X = X.reindex(columns=feature_columns, fill_value=0)
    X[num_cols] = scaler.transform(X[num_cols])
    return X

# ------------------ predict ------------------
if submitted:
    X_input = build_feature_row()

    pred_enc   = classifier.predict(X_input)[0]
    pred_label = label_encoder.inverse_transform([pred_enc])[0]
    probs      = classifier.predict_proba(X_input)[0]
    pred_max_emi = float(regressor.predict(X_input)[0])

    st.subheader("Prediction Results")
    c1, c2 = st.columns(2)

    with c1:
        st.metric("Eligibility", str(pred_label))
        if pred_label == "Eligible":
            st.success(f"Applicant is **{pred_label}** ✅")
        elif pred_label == "High_Risk":
            st.warning(f"Applicant is **{pred_label}** ⚠️")
        else:
            st.error(f"Applicant is **{pred_label}** ❌")

    with c2:
        st.metric("Max Monthly EMI (predicted)", f"₹ {pred_max_emi:,.0f}")

    st.markdown("**Class probabilities**")
    proba_df = (pd.DataFrame({"class": label_encoder.classes_, "probability": probs})
                  .sort_values("probability", ascending=False).reset_index(drop=True))
    st.dataframe(proba_df, use_container_width=True, hide_index=True)
    st.bar_chart(proba_df.set_index("class"))

    with st.expander("Processed feature row sent to the models"):
        st.dataframe(X_input, use_container_width=True)
else:
    st.info("Fill out the form in the sidebar and hit **Predict**.")

Overwriting /content/app.py
